# Scraping for Urban Farms in Dallas County 

In [1]:
import requests
import pandas as pd 

The Dallas Coalition for Hunger Solutions hosts a [map](https://www.dallashunger.org/garden-locator-map) of urban farms, including community gardens, school food gardens, urban farms, market gardens, and teaching gardens. It is based on self-reporting through a form on the page, so it is likely not comprehensive (it doesn't even include Joppy Momma's Farm), but it's  an excellent starting point. 

In [21]:
# Paste the URL you copied from the Network tab here
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'

# Crucial: Add headers so Elfsight doesn't block the request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status() # This will tell you immediately if you got blocked

raw_data = response.json()

# 1. Access the widgets dictionary
widgets_dict = raw_data['data']['widgets']

# 2. Extract the first widget's configuration, bypassing the random ID key
first_widget_config = list(widgets_dict.values())[0]

try:
    # 3. Drill down into the markers
    # In this specific dictionary structure, the map points are nested here:
    markers = first_widget_config['data']['settings']['markers']
    
    # 4. Convert to a DataFrame
    df = pd.DataFrame(markers)
    print(f"Success! Extracted {len(df)} urban farms.")
    
    # 5. Clean up the output to only show the most useful columns
    # Elfsight returns a lot of UI data (like icon colors). We just want the spatial data.
    relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'category'] if col in df.columns]
    
    if relevant_cols:
        clean_df = df[relevant_cols]
        print("\nCleaned Data Sample:")
        print(clean_df.head())
    else:
        # Fallback if the column names are slightly different
        print("\nRaw Data Sample:")
        print(df.head())
        
    # Export the clean geospatial data
    # clean_df.to_csv("dallas_urban_farms.csv", index=False)

except KeyError as e:
    print(f"Missing a key in the final path: {e}")
    print("Keys available in first_widget_config:", first_widget_config.keys())
    if 'data' in first_widget_config:
        print("Keys in 'data':", first_widget_config['data'].keys())

Success! Extracted 191 urban farms.

Cleaned Data Sample:
                               category
0  afb9d952-8c58-405a-9ccc-cc944ec6ad6c
1  a9e8a2fb-c031-484b-8f99-93b1c1debe3b
2  afb9d952-8c58-405a-9ccc-cc944ec6ad6c
3  97d6907d-b8ba-473d-9c4a-a4834c9da7f0
4  afb9d952-8c58-405a-9ccc-cc944ec6ad6c


In [18]:
# Paste the URL you copied from the Network tab here
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'

# Crucial: Add headers so Elfsight doesn't block the request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status() # This will tell you immediately if you got blocked
raw_data = response.json()

# 1. Access the main data block
payload = raw_data['data']

# 2. Inspect the structure of the payload
print("Type of 'data':", type(payload))

if isinstance(payload, dict):
    print("Inner keys available:", payload.keys())
    
    # If you see 'settings' in the keys, look for markers inside it
    if 'settings' in payload:
        print("Keys inside 'settings':", payload['settings'].keys())
        
    # If you see 'widgets' or 'strays' or 'elements', print them
elif isinstance(payload, list):
    print(f"'data' is a list with {len(payload)} items. Printing first item:")
    print(payload[0])


Type of 'data': <class 'dict'>
Inner keys available: dict_keys(['widgets', 'assets'])


In [20]:
url = "https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map..."
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# Recursive function to hunt down the 'markers' key anywhere in the JSON
def find_markers(data, path=""):
    if isinstance(data, dict):
        for key, value in data.items():
            current_path = f"{path}['{key}']" if path else f"['{key}']"
            if key == 'markers':
                return value, current_path
            res, p = find_markers(value, current_path)
            if res is not None:
                return res, p
    elif isinstance(data, list):
        for index, item in enumerate(data):
            current_path = f"{path}[{index}]"
            res, p = find_markers(item, current_path)
            if res is not None:
                return res, p
    return None, None

markers, exact_path = find_markers(raw_data)

if markers:
    print(f"Target found! The exact path is: raw_data{exact_path}")
    print(f"Total locations found: {len(markers)}")
    print("\nSample location data structure:")
    print(markers[0])
else:
    print("Could not find 'markers'. Printing the structure under 'widgets':")
    # If markers doesn't exist, let's see what is inside widgets
    widgets = raw_data['data']['widgets']
    if isinstance(widgets, list):
        print("Widgets is a list. First element keys:", widgets[0].keys() if widgets else "Empty list")
    elif isinstance(widgets, dict):
        print("Widgets is a dict. Keys:", widgets.keys())

Could not find 'markers'. Printing the structure under 'widgets':
Widgets is a dict. Keys: dict_keys([])


In [26]:
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

extracted_markers = []

def extract_markers_programmatically(item):
    """
    Recursively crawls the JSON payload to find any dictionary 
    that represents a physical map marker/location.
    """
    if isinstance(item, dict):
        # Elfsight markers always contain either an address, coordinates, or a marker title
        if 'address' in item or 'location' in item or ('lat' in item and 'lng' in item):
            extracted_markers.append(item)
        else:
            # If it's not a marker itself, dig deeper into its values
            for value in item.values():
                extract_markers_programmatically(value)
                
    elif isinstance(item, list):
        # If it's a list, look at every element in the list
        for element in item:
            extract_markers_programmatically(element)

# Run the crawler on the raw data response
extract_markers_programmatically(raw_data)

# Process and clean the results
if extracted_markers:
    # Convert the collected list of dicts into a DataFrame
    df = pd.DataFrame(extracted_markers)
    
    # Keep only the clean spatial and descriptive data columns, ignoring UI/color configurations
    desired_columns = ['title', 'address', 'lat', 'lng', 'category', 'description', 'phone', 'email', 'website']
    existing_columns = [col for col in desired_columns if col in df.columns]
    
    clean_df = df[existing_columns].copy()
    
    # Remove any duplicate entries that might have been picked up during the crawl
    if 'address' in clean_df.columns:
        clean_df = clean_df.drop_duplicates(subset=['address'])
    elif 'title' in clean_df.columns:
        clean_df = clean_df.drop_duplicates(subset=['title'])
        
    print(f"Success! Dynamically extracted {len(clean_df)} map locations.")
    print("\nHere is a preview of your clean dataset:")
    print(clean_df.head())
    
    # Export it directly to your pipeline
    # clean_df.to_csv("dallas_urban_farms.csv", index=False)
else:
    print("Extraction failed. The data layout did not contain recognized address or coordinate keys.")

Extraction failed. The data layout did not contain recognized address or coordinate keys.


In [28]:
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# 1. Access the widgets dictionary
widgets_dict = raw_data['data']['widgets']

# 2. Extract the first widget's configuration
first_widget_config = list(widgets_dict.values())[0]

# 3. Drill down into the markers
markers = first_widget_config['data']['settings']['markers']

# 4. THE FIX: Orient by index so the hash IDs become rows, and the data becomes columns
df = pd.DataFrame.from_dict(markers, orient='index')

print(f"Success! Extracted {len(df)} urban farms.")

# 5. Clean up the output to only show the geospatial/text data
relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'categoryId'] if col in df.columns]

if relevant_cols:
    clean_df = df[relevant_cols]
    print("\nCleaned Data Sample:")
    print(clean_df.head())
else:
    print("\nRaw Data Columns Available:")
    print(df.columns.tolist())

AttributeError: 'list' object has no attribute 'values'

In [29]:
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# 1. Access the widgets list
widgets_list = raw_data['data']['widgets']

# 2. Grab the first widget from the list
first_widget_config = widgets_list[0]

# 3. Drill down into the markers dictionary
markers = first_widget_config['data']['settings']['markers']

# 4. Orient by index so the hash IDs become rows, and the properties become columns
df = pd.DataFrame.from_dict(markers, orient='index')

print(f"Success! Extracted {len(df)} urban farms.")

# 5. Filter for just the essential spatial and descriptive data
relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'categoryId'] if col in df.columns]

if relevant_cols:
    clean_df = df[relevant_cols]
    print("\nCleaned Data Sample:")
    print(clean_df.head())
    
    # Save the file directly to your workspace
    clean_df.to_csv("dallas_urban_farms.csv", index=False)
    print("\nDataset successfully saved to 'dallas_urban_farms.csv'")
else:
    print("\nRaw Data Columns Available:")
    print(df.columns.tolist())

KeyError: 0

In [2]:
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# 1. Safely extract the widgets container
widgets_container = raw_data['data']['widgets']

# 2. Extract the first widget payload regardless of whether it is a list or a dict
if isinstance(widgets_container, dict):
    # If it's a dict, get the first value safely without using a key like 0
    first_widget_config = next(iter(widgets_container.values()))
elif isinstance(widgets_container, list):
    # If it's a list, look at the first element safely
    first_widget_config = widgets_container[0]
else:
    raise TypeError(f"Unexpected type for widgets: {type(widgets_container)}")

# 3. Drill down into the markers
markers = first_widget_config['data']['settings']['markers']

# 4. Check if markers is a list or a dictionary
if isinstance(markers, dict):
    # If markers is a dictionary, use from_dict orient='index'
    df = pd.DataFrame.from_dict(markers, orient='index')
elif isinstance(markers, list):
    # If markers is already a clean list of dictionaries, load it directly
    df = pd.DataFrame(markers)
else:
    raise TypeError(f"Unexpected type for markers: {type(markers)}")

print(f"Success! Extracted {len(df)} urban farms.")

# 5. Filter for just the essential spatial and descriptive data
relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'categoryId'] if col in df.columns]

if relevant_cols:
    clean_df = df[relevant_cols]
    print("\nCleaned Data Sample:")
    print(clean_df.head())
    clean_df.to_csv("dallas_urban_farms.csv", index=False)
else:
    print("\nColumns found in the dataframe:")
    print(df.columns.tolist())
    print("\nFirst row sample:")
    print(df.iloc[0].to_dict())

Success! Extracted 191 urban farms.

Columns found in the dataframe:
['position', 'coordinates', 'icon', 'category', 'linkUrl', 'infoWindow', 'infoTitle', 'infoDescription', 'infoImage', 'infoAddress', 'infoSite', 'infoPhone', 'infoEmail', 'infoWorkingHours', 'infoWindowOpenedByDefault', 'markerClickAction', 'animation', 'id', 'chosen', 'selected']

First row sample:
{'position': '1513 S. Beltline Rd., Grand Prairie, TX 75052', 'coordinates': '32.7239892, -96.9960085', 'icon': None, 'category': 'afb9d952-8c58-405a-9ccc-cc944ec6ad6c', 'linkUrl': '', 'infoWindow': True, 'infoTitle': 'The Haven Community Garden', 'infoDescription': '<div>38 raised beds; 8 garden rows; 14 fruit trees; hours of operation: dawn till dusk; no annual fee; funded by Keep Grand Prairie Beautiful.</div>', 'infoImage': None, 'infoAddress': '', 'infoSite': '', 'infoPhone': '', 'infoEmail': ' sharon.dehnert@tx.rr.com', 'infoWorkingHours': '', 'infoWindowOpenedByDefault': False, 'markerClickAction': 'infoWindow', 'an

In [3]:
relevant_col = [col for col in ['position', 'coordinates', 'category', 'linkUrl', 'infoTitle', 'infoDescription', 'infoAddress', 'infoSite', 'infoPhone', 'infoEmail'] if col in df.columns]
clean_df = df[relevant_cols]
print(clean_df.head())

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


In [4]:
farms_clean_df = df[['position', 'coordinates', 'category', 'infoTitle', 'infoEmail']]
farms_clean_df.to_csv("dallasUrbanFarms.csv", index=False)